<div style='background:#0a1628;padding:2.5rem 2rem;border-radius:8px;border-left:4px solid #00e5ff;margin-bottom:0.5rem'>
<h1 style='color:#00e5ff;font-family:monospace;margin:0 0 0.4rem;font-size:1.9rem'>⚙️ Django Web Development</h1>
<h2 style='color:#e8f0fe;font-family:monospace;margin:0 0 0.5rem;font-weight:600;font-size:1.3rem'>  Environment Setup Guide</h2>
<p style='color:#8899b4;margin:0 0 0.3rem;font-size:0.9rem'>Run every cell top-to-bottom. Each section validates itself before moving on.</p>
<p style='color:#8899b4;margin:0;font-size:0.85rem'>Django · Generic Django Project Setup</p>
</div>


## What This Notebook Does
| Step | What We Set Up                                                           | Time  |
| ---- | ------------------------------------------------------------------------ | ----- |
| 0    | Pre-flight system checks                                                 | 1 min |
| 1    | Install `virtualenv` (virtual environment manager)                       | 1 min |
| 2    | Create `requirements/` folder and environment-specific requirement files | 1 min |
| 3    | Create `.envs/` folder for each environment                              | 1 min |
| 4    | Install and verify packages                                              | 3 min |
| 5    | Initialize Git repository and add `.gitignore`                           | 1 min |
| 6    | Scaffold project directory structure                                     | 1 min |
| 7    | Full environment validation                                              | 2 min |
| 8    | Understand project files and folder structure                            | 1 min |
| 9    | Docker setup for each environment                                        | 5 min |
| 10   | Final validation and next steps                                          | 2 min |


---
# Step 0 — Pre-Flight System Checks
> Before installing anything, let's confirm your operating system, available RAM, disk space, and network connectivity.


## 0.1 Why These Checks Matter

The masterclass stack has specific requirements:
- **RAM:** Minimum 8 GB (16 GB recommended for smooth performance and incase of integrating LLMs in the future)
- **Disk:** At least 15 GB free — this is for the OS, Python, dependencies, and project files
- **OS:** macOS 12+, Ubuntu 20.04+, or Windows 10/11 (WSL2 recommended)
- **Network:** Required for initial setup only — subsequent work can be done offline

Run the cell below — it will flag any issues immediately.


In [1]:
# ── 0.1 Operating system and Python info ────────────────────────────────
import sys, platform, os

print('╔══════════════════════════════════════════════╗')
print('║         PRE-FLIGHT SYSTEM CHECK              ║')
print('╠══════════════════════════════════════════════╣')
print(f'║  OS:         {platform.system()} {platform.release():<26}║')
print(f'║  Machine:    {platform.machine():<30}║')
print(f'║  Python:     {sys.version.split()[0]:<30}║')
print(f'║  Executable: {sys.executable[:40]:<30}║')
print('╚══════════════════════════════════════════════╝')


╔══════════════════════════════════════════════╗
║         PRE-FLIGHT SYSTEM CHECK              ║
╠══════════════════════════════════════════════╣
║  OS:         Linux 5.4.0-216-generic         ║
║  Machine:    x86_64                        ║
║  Python:     3.13.9                        ║
║  Executable: /home/cheche/anaconda3/bin/python║
╚══════════════════════════════════════════════╝


In [3]:
# ── 0.2 RAM and disk space check ────────────────────────────────────────
import shutil

# Disk space
total, used, free = shutil.disk_usage('/')
free_gb  = free  / (1024**3)
total_gb = total / (1024**3)

# RAM (cross-platform)
ram_gb = None
try:
    import psutil
    ram_gb = psutil.virtual_memory().total / (1024**3)
except ImportError:
    # Fallback: read /proc/meminfo on Linux
    try:
        with open('/proc/meminfo') as f:
            for line in f:
                if 'MemTotal' in line:
                    ram_gb = int(line.split()[1]) / (1024**2)
                    break
    except:
        ram_gb = None

print('=== Hardware Check ===')

# RAM check
if ram_gb:
    ram_ok = ram_gb >= 8
    icon   = '✅' if ram_ok else '⚠️ '
    note   = '' if ram_ok else '  ← Minimum 8 GB recommended'
    print(f'{icon} RAM available: {ram_gb:.1f} GB{note}')
else:
    print('ℹ️  RAM: could not detect (install psutil: pip install psutil)')

# Disk check
disk_ok = free_gb >= 15
icon    = '✅' if disk_ok else '⚠️ '
note    = '' if disk_ok else '  ← Need at least 15 GB free'
print(f'{icon} Disk free:  {free_gb:.1f} GB / {total_gb:.1f} GB total{note}')

if disk_ok and (not ram_gb or ram_gb >= 8):
    print('\n✅ Hardware check passed — good to proceed!')
else:
    print('\n⚠️  Address the warnings above before continuing.')


=== Hardware Check ===
✅ RAM available: 15.5 GB
✅ Disk free:  52.6 GB / 233.7 GB total

✅ Hardware check passed — good to proceed!


In [3]:
# ── 0.3 Network connectivity check ──────────────────────────────────────
import urllib.request, socket

ENDPOINTS = [
    ('pypi.org',        'https://pypi.org'),
    ('ollama.com',      'https://ollama.com'),
    ('github.com',      'https://github.com'),
  
]

print('=== Network Connectivity ===')
all_ok = True
for name, url in ENDPOINTS:
    try:
        urllib.request.urlopen(url, timeout=5)
        print(f'  ✅ {name:<20} reachable')
    except Exception as e:
        print(f'  ❌ {name:<20} FAILED — {str(e)[:50]}')
        all_ok = False

print()
if all_ok:
    print('✅ All endpoints reachable — downloads will work.')
else:
    print('⚠️  Some endpoints unreachable. Check firewall / proxy settings.')


=== Network Connectivity ===
  ✅ pypi.org             reachable
  ✅ ollama.com           reachable
  ✅ github.com           reachable

✅ All endpoints reachable — downloads will work.


In [3]:
#===Check for wifi/signal strength (Linux only)===

import subprocess

def get_wifi_signal_strength():
    try:
        result = subprocess.run(['iwconfig'], capture_output=True, text=True)
        output = result.stdout
        for line in output.splitlines():
            if 'Signal level' in line:
                signal_level = line.split('Signal level=')[1].split()[0]
                return signal_level
    except Exception as e:
        return None


print('=== Wi-Fi Signal Strength ===')
signal_strength = get_wifi_signal_strength()
if signal_strength:
    print(f'  ✅ Wi-Fi signal strength: {signal_strength}')
else:
    print('  ℹ️  Could not detect Wi-Fi signal strength (Linux only).')

=== Wi-Fi Signal Strength ===
  ✅ Wi-Fi signal strength: -45


---
# Step 2 — Python 3.13.9



In [ ]:
# ── 2.1 Verify Python 3.13.9 is available via virtual environment ───────────────────────────
import subprocess, sys

# Check current Python in this notebook
major = sys.version_info.major
minor = sys.version_info.minor

print(f'Current notebook Python: {sys.version}')
print(f'Version tuple: ({major}, {minor})')

if major == 3 and minor == 13:
    print('\n✅ Python 3.13.9 confirmed — correct version!')
elif major == 3 and minor >= 10:
    print(f'\n⚠️  Python 3.{minor} — works but 3.13.9 preferred.')
    print('   To switch: uv python install 3.13.9')
    print('   Then register kernel: uv run python -m ipykernel install --user --name llm-masterclass')
else:
    print(f'\n❌ Python {major}.{minor} is too old. Install 3.13.9:')
    print('    python -m pip install --upgrade pip')


Current notebook Python: 3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:16:10) [GCC 11.2.0]
Version tuple: (3, 13)

✅ Python 3.13.9 confirmed — correct version!


In [5]:
# ── 2.3 Show path to Python 3.8 ────────────────────────────────────────
import subprocess

result = subprocess.run(['uv', 'python', 'find', '3.13.9'], capture_output=True, text=True)
if result.returncode == 0:
    print(f'Python 3.13.9 location: {result.stdout.strip()}')
    print('\n✅ Python 3.13.9 is available.')
else:
    print('Python 3.13.9 not yet installed by uv.')
    print('Run in terminal: uv python install 3.13.9')
    # Auto-install attempt
    print('\nAttempting auto-install...')
    install = subprocess.run(['uv', 'python', 'install', '3.13.9'], capture_output=False, text=True)


Python 3.13.9 location: /home/cheche/anaconda3/bin/python3

✅ Python 3.13.9 is available.


## 3.1 Understanding `requirements` and `virtualenv`

virtual environment is the single source of truth for your project's dependencies.


## 3.2 Installing Dependencies

```bash
# Create requirements folder and environment-specific files
mkdir -p requirements && touch requirements/base.txt requirements/local.txt requirements/production.txt
# Install all dependencies from requirements/base.txt
python3.8 -m venv .venv
source .venv/bin/activate
pip install -r requirements/base.txt
```

In [ ]:
# ── 3.2 Verify virtual environment exists ────────────────────────────────
import sys, pathlib

# Check if we're running inside a uv-managed venv
in_venv = hasattr(sys, 'real_prefix') or (
    hasattr(sys, 'base_prefix') and sys.base_prefix != sys.prefix
)

print(f'Virtual environment active: {in_venv}')
print(f'Python executable: {sys.executable}')
print(f'sys.prefix: {sys.prefix}')

# Show .venv location
venv_path = pathlib.Path(sys.prefix)
site_packages = list(venv_path.glob('lib/python*/site-packages'))
if site_packages:
    pkgs = list(site_packages[0].iterdir())
    print(f'\nsite-packages: {site_packages[0]}')
    print(f'Packages installed: {len(pkgs)}')

if in_venv:
    print('\n✅ Running inside a virtual environment — good.')
else:
    print('\n⚠️  Not in a virtual environment.')
    print('  python -m venv .venv')


Virtual environment active: False
Python executable: /home/cheche/anaconda3/bin/python
sys.prefix: /home/cheche/anaconda3

site-packages: /home/cheche/anaconda3/lib/python3.13/site-packages
Packages installed: 923

⚠️  Not in a virtual environment.
   Start JupyterLab with: uv run jupyter lab


In [7]:
# ── 3.3 Check all core packages are importable ───────────────────────────
import importlib

REQUIRED_PACKAGES = [
    # (import_name,     display_name,          category)
    ('django',          'Django',               'Web Framework'),
    ('openai',          'openai',               'LLM APIs'),
    ('requests',        'requests',             'HTTP Client'),
    ('python_dotenv',   'python-dotenv',        'Environment Variables'),
    ('python-decouple', 'python-decouple',      'Environment Variables'),
    ('pillow',          'Pillow',               'Image Processing'),
]

passed, failed = [], []
for import_name, display, category in REQUIRED_PACKAGES:
    try:
        mod = importlib.import_module(import_name)
        ver = getattr(mod, '__version__', 'installed')
        passed.append((display, ver, category))
    except ImportError as e:
        failed.append((display, str(e), category))

print(f'Package check: {len(passed)}/{len(REQUIRED_PACKAGES)} installed\n')
print(f'{"PACKAGE":<25} {"VERSION":<15} {"CATEGORY"}')
print('─' * 60)
for name, ver, cat in passed:
    print(f'  ✅ {name:<23} {str(ver):<15} {cat}')
if failed:
    print()
    for name, err, cat in failed:
        print(f'  ❌ {name:<23} MISSING          {cat}')
    print()
    print('Fix missing packages:')
    print('  uv sync  (from your project root in terminal)')
else:
    print('\n✅ All required packages are installed!')


Package check: 2/6 installed

PACKAGE                   VERSION         CATEGORY
────────────────────────────────────────────────────────────
  ✅ Django                  4.1.13          Web Framework
  ✅ requests                2.33.1          HTTP Client

  ❌ openai                  MISSING          LLM APIs
  ❌ python-dotenv           MISSING          Environment Variables
  ❌ python-decouple         MISSING          Environment Variables
  ❌ Pillow                  MISSING          Image Processing

Fix missing packages:
  uv sync  (from your project root in terminal)


In [8]:
# ── 3.4 Check package versions are compatible ────────────────────────────
from packaging.version import Version
import importlib

VERSION_REQUIREMENTS = [
    ('openai',    '1.30',  'OpenAI SDK v1+ required for new message format'),
    ('django',    '3.1',   'Django 3.1+ required for project compatibility'),
   
]

print('=== Version Compatibility Check ===')
all_ok = True
for pkg, min_ver, reason in VERSION_REQUIREMENTS:
    try:
        mod = importlib.import_module(pkg)
        actual = getattr(mod, '__version__', '0.0')
        ok = Version(actual) >= Version(min_ver)
        icon = '✅' if ok else '❌'
        print(f'  {icon} {pkg:<12} {actual:<12} (need >= {min_ver})  {"" if ok else "← " + reason}')
        if not ok:
            all_ok = False
    except Exception as e:
        print(f'  ⚠️  {pkg}: {e}')

print()
if all_ok:
    print('✅ All version requirements satisfied!')
else:
    print('Fix with: uv lock --upgrade && uv sync')


=== Version Compatibility Check ===
  ⚠️  openai: No module named 'openai'
  ✅ django       4.1.13       (need >= 3.1)  

✅ All version requirements satisfied!


---
# Step 7 — Scaffolding the Project Structure
> Create the directory layout used throughout all 12 modules.


# 🧪 {PROJECT_NAME} — Django Environment Preflight Check

This notebook verifies that your system is ready for running a **Django project**.  
Only essential checks are included (no unnecessary tooling).

---

## ✅ Step 0 — Project Context

We are setting up:

- **Project name:** `{PROJECT_NAME}`
- **Framework:** Django
- **Purpose:** Backend API
- **Focus:** Environment readiness (not full deployment setup yet)

---

## 🖥️ Step 1 — OS Check

```python
import platform

print("Operating System:", platform.system())
print("OS Version:", platform.version())
print("Machine:", platform.machine())


In [9]:
# ── 7.1 Create full project directory structure ──────────────────────────
import pathlib

PROJECT_NAME = "{PROJECT_NAME}"

DIRECTORIES = [
    f'{PROJECT_NAME}/',
    f'{PROJECT_NAME}/settings/',
    'config/',
    'config/settings/',
    'docker/',
    'docker/local/django/',
    'docker/local/nginx/',
    'docker/local/postgres/',
    'requirements/',
    '.envs/',
    '.envs/.local/',
    'staticfiles/',
    'mediafiles/',
    'core_apps/',
    'core_apps/scripts/',
    'tests/',
]

created = []
existed = []

for d in DIRECTORIES:
    p = pathlib.Path(d)
    if p.exists():
        existed.append(d)
    else:
        p.mkdir(parents=True, exist_ok=True)
        created.append(d)
        # Create .gitkeep so empty dirs are tracked by git
        (p / '.gitkeep').touch()

print(f'Created:  {len(created)} directories')
print(f'Existed:  {len(existed)} directories')
for d in created:
    print(f'  + {d}')
print('\n✅ Project structure ready!')


Created:  5 directories
Existed:  11 directories
  + {PROJECT_NAME}/
  + {PROJECT_NAME}/settings/
  + core_apps/
  + core_apps/scripts/
  + tests/

✅ Project structure ready!


In [9]:
# ── 9.2 List all installed packages with versions ────────────────────────
import subprocess

result = subprocess.run(['uv', 'pip', 'list'], capture_output=True, text=True)
lines  = result.stdout.strip().split('\n')

print(f'Total packages installed: {len(lines) - 2}\n')  # -2 for header rows

# Show first 40 (the most relevant ones)
for line in lines[:42]:
    print(line)

if len(lines) > 42:
    print(f'... and {len(lines) - 42} more packages')


Total packages installed: 86

Package                       Version
----------------------------- ---------
amqp                          5.3.1
anyio                         4.5.2
argon2-cffi                   21.3.0
argon2-cffi-bindings          21.2.0
asgiref                       3.8.1
async-timeout                 5.0.1
backports-zoneinfo            0.2.1
billiard                      3.6.4.0
black                         23.3.0
celery                        5.2.7
certifi                       2026.6.17
cffi                          1.17.1
charset-normalizer            3.4.7
click                         8.1.8
click-didyoumean              0.3.1
click-plugins                 1.1.1.2
click-repl                    0.3.0
coreapi                       2.3.3
coreschema                    0.0.4
cryptography                  47.0.0
defusedxml                    0.7.1
django                        4.1.7
django-allauth                0.52.0
django-appconf                1.0.6
django-autoslu

## ✅ Django Environment Setup Complete ({PROJECT_NAME})

If you've completed all setup steps and validations passed, your **Django development environment is ready**.

This setup uses:

- 🐍 **virtualenv** (Python environment management)
- 🐳 **Docker** (containerized services)
- ⚙️ **Django** (backend framework)

---

### 🧱 What You've Set Up

| Component | Status |
|-----------|--------|
| Python 3.11 | ✅ Installed |
| virtualenv | ✅ Active and configured |
| Django project | ✅ Initialized (`{PROJECT_NAME}`) |
| Dependencies | ✅ Installed via `requirements.txt` |
| `.env` file | ✅ Created (secure config storage) |
| `.gitignore` | ✅ Protects `.env` and venv |
| Docker | ✅ Installed and running |
| Docker Compose | ✅ Ready for services |
| Database service (Postgres/MySQL) | ✅ Containerized |
| Project structure | ✅ Clean Django layout |

---

### 🚀 Ready!

